# Hosting Shotgrid MCP Server on Amazon Bedrock AgentCore Runtime - AWS IAM Inbound Authentication

## Overview

In this tutorial we will learn how to host an Autodesk Shotgrid MCP (Model Context Protocol) server on Amazon Bedrock AgentCore Runtime. We will use the Amazon Bedrock AgentCore Python SDK to wrap Shotgrid API tools as an MCP server compatible with Amazon Bedrock AgentCore.

The Amazon Bedrock AgentCore Python SDK handles the MCP server implementation details so you can focus on your tools' core functionality. It transforms your code into the AgentCore standardized MCP protocol contracts for direct communication.

While the [MCP protocol](https://modelcontextprotocol.io/docs/getting-started/intro) specification traditionally requires OAuth tokens for authentication, AgentCore runtime allows the ability to configure AWS IAM credentials for inbound requests to their MCP servers, addressing a crucial enterprise requirement.

### Tutorial Details

| Information         | Details                                                   |
|:--------------------|:----------------------------------------------------------|
| Tutorial type       | Hosting Tools                                             |
| Tool type           | MCP server for Autodesk Shotgrid                         |
| Tutorial components | Hosting Shotgrid MCP server on AgentCore Runtime         |
| Tutorial vertical   | Media & Entertainment / Production Management            |
| Example complexity  | Intermediate                                              |
| SDK used            | Amazon BedrockAgentCore Python SDK, MCP, Shotgrid API    |

### Tutorial Architecture

In this tutorial we will describe how to deploy a Shotgrid MCP server to AgentCore runtime.

The MCP server provides comprehensive tools for interacting with Autodesk Shotgrid, including:
* Querying projects, shots, assets, tasks, versions, and users
* Creating new shots, assets, tasks, and notes
* Updating entity fields and task statuses
* Searching and retrieving entity information

<div style="text-align:left">
    <img src="images/hosting_mcp_server.png" width="60%"/>
</div>

### Tutorial Key Features

* Creating MCP servers with Shotgrid API integration
* Testing Shotgrid MCP servers locally
* Hosting Shotgrid MCP servers on Amazon Bedrock AgentCore Runtime
* Invoking deployed MCP servers with AWS IAM authentication
* Managing production data through AI-powered tools


## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials configured
* Amazon Bedrock AgentCore SDK
* MCP (Model Context Protocol) library
* Running Docker daemon
* Autodesk Shotgrid account with API access
* Shotgrid Python API (shotgun_api3)
* Shotgrid credentials: SHOTGRID_URL, SHOTGRID_SCRIPT_NAME, SHOTGRID_API_KEY

In [ ]:
# Install dependencies
# Note: You may see a dependency conflict warning about jsonschema-path.
# This is a non-blocking warning from pip's dependency resolver and can be safely ignored.
# The packages will still work correctly despite the version mismatch.
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from bedrock_agentcore_starter_toolkit.operations.runtime import destroy_bedrock_agentcore
from boto3.session import Session
from pathlib import Path
import os

In [ ]:
boto_session = Session()
region = boto_session.region_name

agentcore_control_client = boto_session.client("bedrock-agentcore-control", region_name=region)
ssm_client = boto_session.client('ssm', region_name=region)

tool_name = "mcp_server_iam"

## Setting Up Shotgrid Python API

Before we can deploy the MCP server, we need to set up the Shotgrid Python API. The API is included directly in the project as the `shotgun_api3/` package directory.

### Why Normal Installation Methods Don't Work

We use this unconventional approach because standard installation methods fail in the AgentCore deployment environment:

1. **`pip install git+https://github.com/shotgunsoftware/python-api.git`** ❌
   - The AgentCore SDK uses a minimal Docker base image that doesn't include Git
   - Installing Git would require modifying the Dockerfile, but the SDK regenerates it on each deployment
   - Error: `Git executable not found`

2. **`pip install ./shotgun_api_local`** (local directory with setup.py) ❌
   - The `uv pip install` tool has issues with local directory installations
   - The build fails with exit code 2 during the pip install step

3. **Custom Dockerfile modifications** ❌
   - The AgentCore SDK automatically generates a fresh Dockerfile on each deployment
   - Any manual edits are overwritten

### The Working Solution: Direct Package Import

Instead, we copy the `shotgun_api3/` package directly into the project directory. Python can then import it without any pip installation.

Run these commands to set up the Shotgrid API:

In [ ]:
import os
import shutil
import subprocess

# Check if shotgun_api_local already exists and is valid
needs_clone = True
if os.path.exists('shotgun_api_local'):
    # Check if it's a valid git repo with the shotgun_api3 directory
    if os.path.exists('shotgun_api_local/shotgun_api3'):
        print("ℹ️  shotgun_api_local directory already exists with valid content")
        print("   Skipping git clone step")
        needs_clone = False
    else:
        print("⚠️  shotgun_api_local exists but appears incomplete")
        print("   Removing and re-cloning...")
        shutil.rmtree('shotgun_api_local')

if needs_clone:
    # Clone the Shotgrid Python API repository
    print("📥 Cloning Shotgrid Python API repository...")
    result = subprocess.run(
        ['git', 'clone', 'https://github.com/shotgunsoftware/python-api.git', 'shotgun_api_local'],
        capture_output=True,
        text=True
    )
    
    if result.returncode != 0:
        print("❌ Error cloning repository:")
        print(result.stderr)
        raise RuntimeError(f"Git clone failed with exit code {result.returncode}")
    
    print("✓ Repository cloned successfully")

# Copy the shotgun_api3 package to the current directory
print("\n📦 Copying shotgun_api3 package...")
if os.path.exists('shotgun_api_local/shotgun_api3'):
    # Remove existing shotgun_api3 if it exists
    if os.path.exists('shotgun_api3'):
        shutil.rmtree('shotgun_api3')
        print("   Removed existing shotgun_api3 directory")
    
    shutil.copytree('shotgun_api_local/shotgun_api3', 'shotgun_api3')
    print("✓ Package copied successfully")
else:
    raise FileNotFoundError(
        "shotgun_api_local/shotgun_api3 directory not found. "
        "The git clone may have failed or the repository structure changed."
    )

# Verify the package was copied
if os.path.exists('shotgun_api3') and os.path.isdir('shotgun_api3'):
    print("\n" + "="*60)
    print("✓ Shotgrid Python API successfully set up")
    print("  The shotgun_api3 package is now available for import")
    print("="*60)
else:
    raise RuntimeError("shotgun_api3 directory not found after copy operation")

## Configure Shotgrid Credentials

The MCP server requires Shotgrid API credentials to connect to your Shotgrid instance. Set these environment variables with your Shotgrid credentials:

In [ ]:
import os

# Set Shotgrid credentials as environment variables
os.environ['SHOTGRID_URL'] = "YOUR_SHOTGRID_URL"
os.environ['SHOTGRID_SCRIPT_NAME'] = "YOUR_SCRIPT_NAME"
os.environ['SHOTGRID_API_KEY'] = "YOUR_API_KEY"

print("✓ Shotgrid credentials configured")
print(f"  - URL: {os.environ['SHOTGRID_URL']}")
print(f"  - Script: {os.environ['SHOTGRID_SCRIPT_NAME']}")

## Understanding MCP (Model Context Protocol)

MCP is a protocol that allows AI models to securely access external data and tools. Key concepts:

* **Tools**: Functions that the AI can call to perform actions
* **Streamable HTTP**: Transport protocol used by AgentCore Runtime
* **Session Isolation**: Each client gets isolated sessions via `Mcp-Session-Id` header
* **Stateless Operation**: Servers must support stateless operation for scalability

AgentCore Runtime expects MCP servers to be hosted on `0.0.0.0:8000/mcp` as the default path.

### Project Structure

Let's set up our project with the proper structure:

```
mcp_server_project/
├── mcp_server.py              # Main MCP server code
├── mcp_client.py          # Local testing client
├── mcp_client_remote.py   # Remote testing client
├── requirements.txt          # Dependencies
└── __init__.py              # Python package marker
```

## Creating Shotgrid MCP Server

The Shotgrid MCP server has been created in `mcp_server.py` with comprehensive tools for production management. The server uses FastMCP with `stateless_http=True` which is required for AgentCore Runtime compatibility.

### Available Shotgrid Tools

The MCP server provides the following categories of tools:

**Query Tools:**
* `find_projects` - Find projects in Shotgrid
* `find_shots` - Find shots with optional project filtering
* `find_assets` - Find assets with optional project filtering
* `find_tasks` - Find tasks with optional entity filtering
* `find_versions` - Find versions with optional project filtering
* `find_users` - Find users in Shotgrid
* `find_notes` - Find notes with optional entity filtering

**Create Tools:**
* `create_shot` - Create a new shot in a project
* `create_asset` - Create a new asset in a project
* `create_task` - Create a new task for an entity
* `create_note` - Create a new note linked to entities

**Update Tools:**
* `update_entity` - Update any entity field
* `update_task_status` - Update a task's status

**Utility Tools:**
* `get_entity_by_id` - Get a single entity by ID
* `search_entities` - Search for entities by text
* `get_server_info` - Get Shotgrid server information

### Environment Variables Required

The server requires the following environment variables:
* `SHOTGRID_URL` - Your Shotgrid instance URL
* `SHOTGRID_SCRIPT_NAME` - Script name for API authentication
* `SHOTGRID_API_KEY` - API key for authentication

In [ ]:
%%writefile mcp_server.py
#!/usr/bin/env python3
"""
Shotgrid MCP Server
Provides MCP tools for interacting with Autodesk Shotgrid API
"""

import os
from typing import Any, Optional
from mcp.server.fastmcp import FastMCP

try:
    from shotgun_api3 import Shotgun
except ImportError:
    print("Error: shotgun_api3 not found. Please ensure the Shotgrid Python API is installed.")
    import sys
    sys.exit(1)

# Initialize FastMCP server
mcp = FastMCP("Shotgrid MCP Server", host="0.0.0.0", stateless_http=True)

# Global Shotgrid connection
sg: Optional[Shotgun] = None


def get_shotgrid_connection() -> Shotgun:
    """Get or create Shotgrid connection using environment variables."""
    global sg
    if sg is None:
        shotgrid_url = os.getenv("SHOTGRID_URL")
        script_name = os.getenv("SHOTGRID_SCRIPT_NAME")
        api_key = os.getenv("SHOTGRID_API_KEY")
        
        # Debug: Print what we found in environment
        print(f"[DEBUG] SHOTGRID_URL from env: {shotgrid_url}")
        print(f"[DEBUG] SHOTGRID_SCRIPT_NAME from env: {script_name}")
        print(f"[DEBUG] SHOTGRID_API_KEY from env: {'SET' if api_key else 'NOT SET'}")
        
        if not all([shotgrid_url, script_name, api_key]):
            raise ValueError(
                "Missing required environment variables: SHOTGRID_URL, "
                "SHOTGRID_SCRIPT_NAME, SHOTGRID_API_KEY"
            )
        
        sg = Shotgun(shotgrid_url, script_name=script_name, api_key=api_key)
    
    return sg


# ============================================================================
# QUERY TOOLS
# ============================================================================

@mcp.tool()
def find_projects(
    filters: Optional[list] = None,
    fields: Optional[list] = None,
    limit: int = 100
) -> list[dict[str, Any]]:
    """
    Find projects in Shotgrid.
    
    Args:
        filters: Optional list of filter conditions (e.g., [['name', 'is', 'MyProject']])
        fields: Optional list of fields to return (default: ['id', 'name', 'code'])
        limit: Maximum number of results (default: 100)
    
    Returns:
        List of project dictionaries
    """
    sg = get_shotgrid_connection()
    if fields is None:
        fields = ['id', 'name', 'code', 'sg_status']
    
    return sg.find('Project', filters or [], fields, limit=limit)


@mcp.tool()
def find_shots(
    project_id: Optional[int] = None,
    filters: Optional[list] = None,
    fields: Optional[list] = None,
    limit: int = 100
) -> list[dict[str, Any]]:
    """
    Find shots in Shotgrid.
    
    Args:
        project_id: Optional project ID to filter by
        filters: Optional list of filter conditions
        fields: Optional list of fields to return
        limit: Maximum number of results (default: 100)
    
    Returns:
        List of shot dictionaries
    """
    sg = get_shotgrid_connection()
    if fields is None:
        fields = ['id', 'code', 'sg_status_list', 'project']
    
    query_filters = filters or []
    if project_id:
        query_filters.append(['project', 'is', {'type': 'Project', 'id': project_id}])
    
    return sg.find('Shot', query_filters, fields, limit=limit)


@mcp.tool()
def find_assets(
    project_id: Optional[int] = None,
    filters: Optional[list] = None,
    fields: Optional[list] = None,
    limit: int = 100
) -> list[dict[str, Any]]:
    """
    Find assets in Shotgrid.
    
    Args:
        project_id: Optional project ID to filter by
        filters: Optional list of filter conditions
        fields: Optional list of fields to return
        limit: Maximum number of results (default: 100)
    
    Returns:
        List of asset dictionaries
    """
    sg = get_shotgrid_connection()
    if fields is None:
        fields = ['id', 'code', 'sg_asset_type', 'sg_status_list', 'project']
    
    query_filters = filters or []
    if project_id:
        query_filters.append(['project', 'is', {'type': 'Project', 'id': project_id}])
    
    return sg.find('Asset', query_filters, fields, limit=limit)


@mcp.tool()
def find_tasks(
    entity_type: Optional[str] = None,
    entity_id: Optional[int] = None,
    filters: Optional[list] = None,
    fields: Optional[list] = None,
    limit: int = 100
) -> list[dict[str, Any]]:
    """
    Find tasks in Shotgrid.
    
    Args:
        entity_type: Optional entity type to filter by (e.g., 'Shot', 'Asset')
        entity_id: Optional entity ID to filter by
        filters: Optional list of filter conditions
        fields: Optional list of fields to return
        limit: Maximum number of results (default: 100)
    
    Returns:
        List of task dictionaries
    """
    sg = get_shotgrid_connection()
    if fields is None:
        fields = ['id', 'content', 'sg_status_list', 'entity', 'task_assignees']
    
    query_filters = filters or []
    if entity_type and entity_id:
        query_filters.append(['entity', 'is', {'type': entity_type, 'id': entity_id}])
    
    return sg.find('Task', query_filters, fields, limit=limit)


@mcp.tool()
def find_versions(
    project_id: Optional[int] = None,
    filters: Optional[list] = None,
    fields: Optional[list] = None,
    limit: int = 100
) -> list[dict[str, Any]]:
    """
    Find versions in Shotgrid.
    
    Args:
        project_id: Optional project ID to filter by
        filters: Optional list of filter conditions
        fields: Optional list of fields to return
        limit: Maximum number of results (default: 100)
    
    Returns:
        List of version dictionaries
    """
    sg = get_shotgrid_connection()
    if fields is None:
        fields = ['id', 'code', 'sg_status_list', 'entity', 'user', 'created_at']
    
    query_filters = filters or []
    if project_id:
        query_filters.append(['project', 'is', {'type': 'Project', 'id': project_id}])
    
    return sg.find('Version', query_filters, fields, limit=limit)


@mcp.tool()
def find_users(
    filters: Optional[list] = None,
    fields: Optional[list] = None,
    limit: int = 100
) -> list[dict[str, Any]]:
    """
    Find users in Shotgrid.
    
    Args:
        filters: Optional list of filter conditions
        fields: Optional list of fields to return
        limit: Maximum number of results (default: 100)
    
    Returns:
        List of user dictionaries
    """
    sg = get_shotgrid_connection()
    if fields is None:
        fields = ['id', 'login', 'name', 'email', 'sg_status_list']
    
    return sg.find('HumanUser', filters or [], fields, limit=limit)


@mcp.tool()
def find_notes(
    entity_type: Optional[str] = None,
    entity_id: Optional[int] = None,
    filters: Optional[list] = None,
    fields: Optional[list] = None,
    limit: int = 100
) -> list[dict[str, Any]]:
    """
    Find notes in Shotgrid.
    
    Args:
        entity_type: Optional entity type to filter by
        entity_id: Optional entity ID to filter by
        filters: Optional list of filter conditions
        fields: Optional list of fields to return
        limit: Maximum number of results (default: 100)
    
    Returns:
        List of note dictionaries
    """
    sg = get_shotgrid_connection()
    if fields is None:
        fields = ['id', 'subject', 'content', 'note_links', 'user', 'created_at']
    
    query_filters = filters or []
    if entity_type and entity_id:
        query_filters.append(['note_links', 'is', {'type': entity_type, 'id': entity_id}])
    
    return sg.find('Note', query_filters, fields, limit=limit)


# ============================================================================
# CREATE TOOLS
# ============================================================================

@mcp.tool()
def create_shot(
    project_id: int,
    code: str,
    description: Optional[str] = None,
    additional_fields: Optional[dict] = None
) -> dict[str, Any]:
    """
    Create a new shot in Shotgrid.
    
    Args:
        project_id: ID of the project
        code: Shot code/name
        description: Optional shot description
        additional_fields: Optional dictionary of additional fields
    
    Returns:
        Created shot dictionary
    """
    sg = get_shotgrid_connection()
    data = {
        'project': {'type': 'Project', 'id': project_id},
        'code': code
    }
    
    if description:
        data['description'] = description
    
    if additional_fields:
        data.update(additional_fields)
    
    return sg.create('Shot', data)


@mcp.tool()
def create_asset(
    project_id: int,
    code: str,
    asset_type: str,
    description: Optional[str] = None,
    additional_fields: Optional[dict] = None
) -> dict[str, Any]:
    """
    Create a new asset in Shotgrid.
    
    Args:
        project_id: ID of the project
        code: Asset code/name
        asset_type: Type of asset (e.g., 'Character', 'Prop', 'Environment')
        description: Optional asset description
        additional_fields: Optional dictionary of additional fields
    
    Returns:
        Created asset dictionary
    """
    sg = get_shotgrid_connection()
    data = {
        'project': {'type': 'Project', 'id': project_id},
        'code': code,
        'sg_asset_type': asset_type
    }
    
    if description:
        data['description'] = description
    
    if additional_fields:
        data.update(additional_fields)
    
    return sg.create('Asset', data)


@mcp.tool()
def create_task(
    entity_type: str,
    entity_id: int,
    content: str,
    task_assignees: Optional[list] = None,
    additional_fields: Optional[dict] = None
) -> dict[str, Any]:
    """
    Create a new task in Shotgrid.
    
    Args:
        entity_type: Type of entity (e.g., 'Shot', 'Asset')
        entity_id: ID of the entity
        content: Task name/content
        task_assignees: Optional list of user IDs to assign
        additional_fields: Optional dictionary of additional fields
    
    Returns:
        Created task dictionary
    """
    sg = get_shotgrid_connection()
    data = {
        'entity': {'type': entity_type, 'id': entity_id},
        'content': content
    }
    
    if task_assignees:
        data['task_assignees'] = [{'type': 'HumanUser', 'id': uid} for uid in task_assignees]
    
    if additional_fields:
        data.update(additional_fields)
    
    return sg.create('Task', data)


@mcp.tool()
def create_note(
    subject: str,
    content: str,
    note_links: list[dict],
    user_id: Optional[int] = None,
    additional_fields: Optional[dict] = None
) -> dict[str, Any]:
    """
    Create a new note in Shotgrid.
    
    Args:
        subject: Note subject
        content: Note content/body
        note_links: List of entities to link (e.g., [{'type': 'Shot', 'id': 123}])
        user_id: Optional user ID (defaults to script user)
        additional_fields: Optional dictionary of additional fields
    
    Returns:
        Created note dictionary
    """
    sg = get_shotgrid_connection()
    data = {
        'subject': subject,
        'content': content,
        'note_links': note_links
    }
    
    if user_id:
        data['user'] = {'type': 'HumanUser', 'id': user_id}
    
    if additional_fields:
        data.update(additional_fields)
    
    return sg.create('Note', data)


# ============================================================================
# UPDATE TOOLS
# ============================================================================

@mcp.tool()
def update_entity(
    entity_type: str,
    entity_id: int,
    data: dict[str, Any]
) -> dict[str, Any]:
    """
    Update an entity in Shotgrid.
    
    Args:
        entity_type: Type of entity (e.g., 'Shot', 'Asset', 'Task')
        entity_id: ID of the entity
        data: Dictionary of fields to update
    
    Returns:
        Updated entity dictionary
    """
    sg = get_shotgrid_connection()
    return sg.update(entity_type, entity_id, data)


@mcp.tool()
def update_task_status(
    task_id: int,
    status: str
) -> dict[str, Any]:
    """
    Update a task's status in Shotgrid.
    
    Args:
        task_id: ID of the task
        status: New status (e.g., 'ip', 'fin', 'wtg')
    
    Returns:
        Updated task dictionary
    """
    sg = get_shotgrid_connection()
    return sg.update('Task', task_id, {'sg_status_list': status})


# ============================================================================
# UTILITY TOOLS
# ============================================================================

@mcp.tool()
def get_entity_by_id(
    entity_type: str,
    entity_id: int,
    fields: Optional[list] = None
) -> Optional[dict[str, Any]]:
    """
    Get a single entity by ID.
    
    Args:
        entity_type: Type of entity (e.g., 'Shot', 'Asset', 'Task')
        entity_id: ID of the entity
        fields: Optional list of fields to return
    
    Returns:
        Entity dictionary or None if not found
    """
    sg = get_shotgrid_connection()
    return sg.find_one(entity_type, [['id', 'is', entity_id]], fields or ['id'])


@mcp.tool()
def search_entities(
    entity_type: str,
    search_text: str,
    fields: Optional[list] = None,
    limit: int = 50
) -> list[dict[str, Any]]:
    """
    Search for entities by text across multiple fields.
    
    Args:
        entity_type: Type of entity to search (e.g., 'Shot', 'Asset')
        search_text: Text to search for
        fields: Optional list of fields to return
        limit: Maximum number of results (default: 50)
    
    Returns:
        List of matching entity dictionaries
    """
    sg = get_shotgrid_connection()
    # Use text_search for broad searching
    filters = [['code', 'contains', search_text]]
    return sg.find(entity_type, filters, fields or ['id', 'code'], limit=limit)


@mcp.tool()
def get_server_info() -> dict[str, Any]:
    """
    Get Shotgrid server information.
    
    Returns:
        Dictionary with server version and other info
    """
    sg = get_shotgrid_connection()
    return sg.server_info


if __name__ == "__main__":
    # Print startup information
    print("=" * 60)
    print("SHOTGRID MCP SERVER STARTING")
    print("=" * 60)
    print(f"SHOTGRID_URL: {os.getenv('SHOTGRID_URL', 'NOT SET')}")
    print(f"SHOTGRID_SCRIPT_NAME: {os.getenv('SHOTGRID_SCRIPT_NAME', 'NOT SET')}")
    print(f"SHOTGRID_API_KEY: {'SET' if os.getenv('SHOTGRID_API_KEY') else 'NOT SET'}")
    print("=" * 60)
    print()
    
    # Run the MCP server
    mcp.run(transport="streamable-http")

### What This Code Does

* **FastMCP**: Creates an MCP server that can host your tools
* **@mcp.tool()**: Decorator that turns your Python functions into MCP tools
* **stateless_http=True**: Required for AgentCore Runtime compatibility
* **Tools**: Three simple tools demonstrating different types of operations

## Creating Local Testing Client

Before deploying to AgentCore Runtime, let's create a client to test our MCP server locally:

In [ ]:
%%writefile mcp_client.py
import asyncio

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    mcp_url = "http://localhost:8000/mcp"
    headers = {}

    async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tool_result = await session.list_tools()
            print("Available tools:")
            for tool in tool_result.tools:
                print(f"  - {tool.name}: {tool.description}")

if __name__ == "__main__":
    asyncio.run(main())

 ### Testing Locally

To test your MCP server locally:

1. **Terminal 1**: Set Shotgrid credentials and start the MCP server
   ```bash
   export SHOTGRID_URL="YOUR_SHOTGRID_URL"
   export SHOTGRID_SCRIPT_NAME="YOUR_SCRIPT_NAME"
   export SHOTGRID_API_KEY="YOUR_API_KEY"
   python mcp_server.py
   ```
   
2. **Terminal 2**: Run the test client
   ```bash
   python mcp_client.py
   ```

You should see 16 Shotgrid tools found in the output.

**Note**: The Shotgrid credentials must be set in the terminal where you run the MCP server, as environment variables set in Jupyter notebook cells don't automatically propagate to separate terminal processes.

### Invoking Shotgrid Tools Locally

Now let's verify that the MCP server can actually execute Shotgrid tools by invoking them locally - this will test the connection to your Shotgrid instance and verify the tools work correctly before deploying to AgentCore Runtime.

**Important**: Before running the invoke script, make sure your MCP server is running in Terminal 1 with the Shotgrid credentials exported:

In [ ]:
%%writefile mcp_client_interaction.py
import asyncio
import json
import os
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def test_shotgrid_tools():
    """Test Shotgrid MCP tools by invoking them locally."""
    
    # Debug: Print Shotgrid credentials from environment
    print("\n" + "="*60)
    print("DEBUG: Checking Shotgrid Environment Variables")
    print("="*60)
    print(f"SHOTGRID_URL: {os.getenv('SHOTGRID_URL', 'NOT SET')}")
    print(f"SHOTGRID_SCRIPT_NAME: {os.getenv('SHOTGRID_SCRIPT_NAME', 'NOT SET')}")
    print(f"SHOTGRID_API_KEY: {os.getenv('SHOTGRID_API_KEY', 'NOT SET')}")
    print("="*60 + "\n")
    
    # Connect to the already-running local MCP server
    mcp_url = "http://localhost:8000/mcp"
    headers = {}
    
    async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            # Initialize the connection
            await session.initialize()
            
            print("\n" + "="*60)
            print("SHOTGRID MCP SERVER - LOCAL TOOL INVOCATION TEST")
            print("="*60 + "\n")
            
            # List available tools
            tools = await session.list_tools()
            print(f"✓ Connected to MCP server at {mcp_url}")
            print(f"✓ Found {len(tools.tools)} Shotgrid tools\n")
            
            # Test 1: Get server info
            print("Test 1: Getting Shotgrid server information...")
            try:
                result = await session.call_tool("get_server_info", arguments={})
                print("✓ Server info retrieved successfully")
                if result.content:
                    for content in result.content:
                        if hasattr(content, 'text') and content.text:
                            try:
                                data = json.loads(content.text)
                                print(f"  - Server version: {data.get('version', 'N/A')}")
                                print(f"  - Server URL: {data.get('url', 'N/A')}")
                            except json.JSONDecodeError:
                                print(f"  - Response (text): {content.text[:200]}")
                        else:
                            print(f"  - Response type: {type(content).__name__}")
                            print(f"  - Response: {str(content)[:200]}")
            except Exception as e:
                print(f"✗ Error: {str(e)}")
            
            print()
            
            # Test 2: Find projects
            print("Test 2: Finding Shotgrid projects...")
            try:
                result = await session.call_tool(
                    "find_projects",
                    arguments={"limit": 5}
                )
                print("✓ Projects retrieved successfully")
                if result.content:
                    projects = []
                    for content in result.content:
                        if hasattr(content, 'text') and content.text:
                            try:
                                data = json.loads(content.text)
                                # Each content item is a single project dict
                                if isinstance(data, dict):
                                    projects.append(data)
                                elif isinstance(data, list):
                                    projects.extend(data)
                            except json.JSONDecodeError:
                                print(f"  - Response (text): {content.text[:500]}")
                    
                    # Print results AFTER collecting all items
                    print(f"  - Found {len(projects)} project(s)")
                    for proj in projects[:5]:  # Show first 5
                        print(f"    • {proj.get('name', 'Unnamed')} (ID: {proj.get('id')}, Status: {proj.get('sg_status', 'N/A')})")
            except Exception as e:
                print(f"✗ Error: {str(e)}")
            
            print()
            
            # Test 3: Find users
            print("Test 3: Finding Shotgrid users...")
            try:
                result = await session.call_tool(
                    "find_users",
                    arguments={"limit": 5}
                )
                print("✓ Users retrieved successfully")
                if result.content:
                    users = []
                    for content in result.content:
                        if hasattr(content, 'text') and content.text:
                            try:
                                data = json.loads(content.text)
                                # Each content item is a single user dict
                                if isinstance(data, dict):
                                    users.append(data)
                                elif isinstance(data, list):
                                    users.extend(data)
                            except json.JSONDecodeError:
                                print(f"  - Response (text): {content.text[:500]}")
                    
                    # Print results AFTER collecting all items
                    print(f"  - Found {len(users)} user(s)")
                    for user in users[:5]:  # Show first 5
                        print(f"    • {user.get('name', 'Unnamed')} (Login: {user.get('login', 'N/A')}, Status: {user.get('sg_status_list', 'N/A')})")
            except Exception as e:
                print(f"✗ Error: {str(e)}")
            
            print()
            
            # Test 4: Check schema to see what entities exist
            print("Test 4: Checking Shotgrid schema...")
            try:
                # Try to get schema information
                result = await session.call_tool("get_server_info", arguments={})
                if result.content:
                    for content in result.content:
                        if hasattr(content, 'text') and content.text:
                            data = json.loads(content.text)
                            print(f"  - Connected to: {data.get('url', 'N/A')}")
                            print(f"  - Version: {data.get('version', 'N/A')}")
                            print(f"\n  ✓ Tutorial working correctly!")
                            print(f"    The MCP server successfully connected to Shotgrid")
                            print(f"    and retrieved real production data.")
            except Exception as e:
                print(f"✗ Error: {str(e)}")
            
            print("\n" + "="*60)
            print("LOCAL TOOL INVOCATION TEST COMPLETE")
            print("="*60 + "\n")

if __name__ == "__main__":
    asyncio.run(test_shotgrid_tools())

In [ ]:
# Run the local tool invocation test
!python mcp_client_interaction.py

## Configuring AgentCore Runtime Deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [ ]:
print(f"Using AWS region: {region}")

required_files = ["mcp_server.py", "requirements.txt"]
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found ✓")

agentcore_runtime = Runtime()

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="mcp_server.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    protocol="MCP",
    agent_name=tool_name,
)
print("Configuration completed ✓")

## Launching MCP Server to AgentCore Runtime

Now that we've got a docker file, let's launch the MCP server to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

### Passing Environment Variables to AgentCore Runtime

The MCP server requires Shotgrid credentials to connect to your Shotgrid instance. We'll pass these credentials as environment variables to the AgentCore Runtime deployment using the `env_vars` parameter in the `launch()` method.

These environment variables will be available to the MCP server when it runs in the AgentCore Runtime container:
* `SHOTGRID_URL` - Your Shotgrid instance URL
* `SHOTGRID_SCRIPT_NAME` - Script name for API authentication
* `SHOTGRID_API_KEY` - API key for authentication

**Note**: In this tutorial, we're passing credentials directly for simplicity. For production deployments, consider using AWS Secrets Manager to securely store and retrieve credentials.

In [ ]:
print("Launching MCP server to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch(
    env_vars={
        "SHOTGRID_URL": "YOUR_SHOTGRID_URL",
        "SHOTGRID_SCRIPT_NAME": "YOUR_SCRIPT_NAME",
        "SHOTGRID_API_KEY": "YOUR_API_KEY"
    }
)
print("Launch completed ✓")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

In [ ]:

agent_arn_response = ssm_client.put_parameter(
    Name='/mcp_server/runtime_iam/agent_arn',
    Value=launch_result.agent_arn,
    Type='String',
    Description='Agent ARN for MCP server with inbound auth',
    Overwrite=True
)
print("✓ Agent ARN stored in Parameter Store")

print("\nConfiguration stored successfully!")
print(f"Agent ARN: {launch_result.agent_arn}")

## Creating Remote Testing Client

Now let's create a client to test our deployed MCP server. This client will retrieve the necessary credentials from AWS and connect to the deployed server:

In [ ]:
%%writefile mcp_client_remote.py       
import asyncio
import json
import sys
import logging
import boto3
from boto3.session import Session
from mcp import ClientSession
from streamable_http_sigv4 import streamablehttp_client_with_sigv4

# Configure logging to suppress the response ID normalization warning
logging.basicConfig(level=logging.ERROR)
logging.getLogger('httpx').setLevel(logging.WARNING)


def create_streamable_http_transport_sigv4(
    mcp_url: str, service_name: str, region: str, timeout: int = 300
):
    """Create a streamable HTTP transport with AWS SigV4 authentication."""
    session = boto3.Session()
    credentials = session.get_credentials()

    return streamablehttp_client_with_sigv4(
        url=mcp_url,
        credentials=credentials,
        service=service_name,
        region=region,
        timeout=timeout,
        terminate_on_close=False,
    )


async def main():
    """Test Shotgrid MCP tools by invoking them remotely."""
    boto_session = Session()
    region = boto_session.region_name
    print(f"Using AWS region: {region}")

    ssm_client = boto3.client("ssm", region_name=region)

    try:
        agent_arn_response = ssm_client.get_parameter(
            Name="/mcp_server/runtime_iam/agent_arn"
        )
        agent_arn = agent_arn_response["Parameter"]["Value"]
        print(f"Retrieved Agent ARN: {agent_arn}")
    except Exception as e:
        print(f"❌ Error retrieving Agent ARN: {e}")
        sys.exit(1)

    if not agent_arn:
        print("❌ Error: AGENT_ARN not found")
        sys.exit(1)

    encoded_arn = agent_arn.replace(":", "%3A").replace("/", "%2F")
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    print(f"Connecting to: {mcp_url}\n")

    try:
        async with create_streamable_http_transport_sigv4(
            mcp_url=mcp_url, service_name="bedrock-agentcore", region=region, timeout=300
        ) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")

                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()

                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}")
                    print(f"   Description: {tool.description}")
                    if hasattr(tool, "inputSchema") and tool.inputSchema:
                        properties = tool.inputSchema.get("properties", {})
                        if properties:
                            print(f"   Parameters: {list(properties.keys())}")
                    print()

                print(f"✅ Successfully connected to MCP server!")
                print(f"Found {len(tool_result.tools)} tools available.")

    except Exception as e:
        print(f"\n❌ Error connecting to MCP server: {e}")
        import traceback
        print("\n🔍 Full error traceback:")
        traceback.print_exc()
        sys.exit(1)


if __name__ == "__main__":
    asyncio.run(main())


## Testing Your Deployed MCP Server

Let's test our deployed MCP server using the remote client:

In [ ]:
print("Testing deployed MCP server...")
print("=" * 50)
!python mcp_client_remote.py

### Invoking MCP Tools Remotely

Now let's create an enhanced client that not only lists tools but also invokes them to demonstrate the full MCP functionality:

In [ ]:
%%writefile invoke_mcp_tools.py
import asyncio
import sys
import os
import json
import logging
import boto3
from boto3.session import Session
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from streamable_http_sigv4 import streamablehttp_client_with_sigv4

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


def create_streamable_http_transport_sigv4(
    mcp_url: str, service_name: str, region: str
):
    """
    Create a streamable HTTP transport with AWS SigV4 authentication.

    This function creates an MCP client transport that uses AWS Signature Version 4 (SigV4)
    to authenticate requests. This is necessary because standard MCP clients don't natively
    support AWS IAM authentication, and this bridges that gap.

    Args:
        mcp_url (str): The URL of the MCP gateway endpoint
        service_name (str): The AWS service name for SigV4 signing (typically "bedrock-agentcore")
        region (str): The AWS region where the gateway is deployed

    Returns:
        StreamableHTTPTransportWithSigV4: A transport instance configured for SigV4 auth

    Example:
        >>> transport = create_streamable_http_transport_sigv4(
        ...     mcp_url=".../mcp",
        ...     service_name="bedrock-agentcore",
        ...     region="us-west-2"
        ... )
    """
    # Get AWS credentials from the current boto3 session
    # These credentials will be used to sign requests with SigV4
    session = boto3.Session()
    credentials = session.get_credentials()

    # Create and return the custom transport with SigV4 signing capability
    return streamablehttp_client_with_sigv4(
        url=mcp_url,
        credentials=credentials,
        service=service_name,
        region=region,
    )


def get_full_tools_list(client):
    """
    Retrieve the complete list of tools from an MCP client, handling pagination.

    MCP servers may return tools in paginated responses. This function handles the
    pagination automatically and returns all available tools in a single list.

    Args:
        client: An MCP client instance (from strands.tools.mcp.mcp_client.MCPClient)

    Returns:
        list: A complete list of all tools available from the MCP server

    Example:
        >>> mcp_client = MCPClient(lambda: create_transport())
        >>> all_tools = get_full_tools_list(mcp_client)
        >>> print(f"Found {len(all_tools)} tools")
    """
    more_tools = True
    tools = []
    pagination_token = None

    # Loop until we've fetched all pages
    while more_tools:
        tmp_tools = client.list_tools_sync(pagination_token=pagination_token)

        tools.extend(tmp_tools)

        # Check if there are more pages to fetch
        if tmp_tools.pagination_token is None:
            # No more pages - we're done
            more_tools = False
        else:
            # More pages exist - prepare to fetch the next one
            more_tools = True
            pagination_token = tmp_tools.pagination_token

    return tools


async def main():
    boto_session = Session()
    region = boto_session.region_name
    print(f"Using AWS region: {region}")

    ssm_client = boto3.client("ssm", region_name=region)

    agent_arn_response = ssm_client.get_parameter(
        Name="/mcp_server/runtime_iam/agent_arn"
    )
    agent_arn = agent_arn_response["Parameter"]["Value"]
    print(f"Retrieved Agent ARN: {agent_arn}")

    if not agent_arn:
        print("❌ Error: AGENT_ARN not found")
        sys.exit(1)

    encoded_arn = agent_arn.replace(":", "%3A").replace("/", "%2F")
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"

    try:
        async with create_streamable_http_transport_sigv4(
                mcp_url=mcp_url, service_name="bedrock-agentcore", region=region
        ) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")

                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()

                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}: {tool.description}")

                print("\n🧪 Testing MCP Tools:")
                print("=" * 50)

                # Test 1: Get server info
                print("Test 1: Getting Shotgrid server information...")
                try:
                    result = await asyncio.wait_for(
                        session.call_tool("get_server_info", arguments={}),
                        timeout=30
                    )
                    print("✓ Server info retrieved successfully")
                    if result.content:
                        for content in result.content:
                            if hasattr(content, 'text') and content.text:
                                try:
                                    data = json.loads(content.text)
                                    print(f"  - Server version: {data.get('version', 'N/A')}")
                                    print(f"  - Server URL: {data.get('url', 'N/A')}")
                                except json.JSONDecodeError:
                                    print(f"  - Response (text): {content.text[:200]}")
                except asyncio.TimeoutError:
                    print("✗ Timeout (30s)")
                except Exception as e:
                    print(f"✗ Error: {str(e)}")

                print()

                # Test 2: Find projects
                print("Test 2: Finding Shotgrid projects...")
                try:
                    result = await asyncio.wait_for(
                        session.call_tool("find_projects", arguments={"limit": 5}),
                        timeout=30
                    )
                    print("✓ Projects retrieved successfully")
                    if result.content:
                        projects = []
                        for content in result.content:
                            if hasattr(content, 'text') and content.text:
                                try:
                                    data = json.loads(content.text)
                                    if isinstance(data, dict):
                                        projects.append(data)
                                    elif isinstance(data, list):
                                        projects.extend(data)
                                except json.JSONDecodeError:
                                    print(f"  - Response (text): {content.text[:500]}")

                        print(f"  - Found {len(projects)} project(s)")
                        for proj in projects[:5]:
                            print(f"    • {proj.get('name', 'Unnamed')} (ID: {proj.get('id')}, Status: {proj.get('sg_status', 'N/A')})")
                except asyncio.TimeoutError:
                    print("✗ Timeout (30s)")
                except Exception as e:
                    print(f"✗ Error: {str(e)}")

                print()

                # Test 3: Find users
                print("Test 3: Finding Shotgrid users...")
                try:
                    result = await asyncio.wait_for(
                        session.call_tool("find_users", arguments={"limit": 5}),
                        timeout=30
                    )
                    print("✓ Users retrieved successfully")
                    if result.content:
                        users = []
                        for content in result.content:
                            if hasattr(content, 'text') and content.text:
                                try:
                                    data = json.loads(content.text)
                                    if isinstance(data, dict):
                                        users.append(data)
                                    elif isinstance(data, list):
                                        users.extend(data)
                                except json.JSONDecodeError:
                                    print(f"  - Response (text): {content.text[:500]}")

                        print(f"  - Found {len(users)} user(s)")
                        for user in users[:5]:
                            print(f"    • {user.get('name', 'Unnamed')} (Login: {user.get('login', 'N/A')}, Status: {user.get('sg_status_list', 'N/A')})")
                except asyncio.TimeoutError:
                    print("✗ Timeout (30s)")
                except Exception as e:
                    print(f"✗ Error: {str(e)}")

                print()

                # Test 4: Verify connection
                print("Test 4: Verifying Shotgrid connection...")
                try:
                    result = await asyncio.wait_for(
                        session.call_tool("get_server_info", arguments={}),
                        timeout=30
                    )
                    if result.content:
                        for content in result.content:
                            if hasattr(content, 'text') and content.text:
                                data = json.loads(content.text)
                                print(f"  - Connected to: {data.get('url', 'N/A')}")
                                print(f"  - Version: {data.get('version', 'N/A')}")
                                print(f"\n  ✓ Tutorial working correctly!")
                                print(f"    The MCP server successfully connected to Shotgrid")
                                print(f"    and retrieved real production data.")
                except asyncio.TimeoutError:
                    print("✗ Timeout (30s)")
                except Exception as e:
                    print(f"✗ Error: {str(e)}")

                print("\n" + "="*60)
                print("REMOTE TOOL INVOCATION TEST COMPLETE")
                print("="*60 + "\n")

    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        import traceback

        print("\n🔍 Full error traceback:")
        traceback.print_exc()
        sys.exit(1)


if __name__ == "__main__":
    asyncio.run(main())


## Test Tool Invocation

Let's test our MCP tools by actually invoking them:

In [ ]:
print("Testing MCP tool invocation...")
print("=" * 50)
!python invoke_mcp_tools.py

## Testing remote MCP server - Boto3 Approach 

This section demonstrates an alternative approach to testing the deployed MCP server using the Boto3 SDK's `invoke_agent_runtime` API. The SDK handles AWS SigV4 request signing automatically, simplifying IAM authentication for runtime invocations

### Create Remote Testing Client - Boto3

Let's create a client to test our deployed MCP server using the Boto3 API: 

In [ ]:
%%writefile mcp_client_remote_boto3.py   

import boto3
import json
import traceback
from boto3.session import Session
from botocore.exceptions import ClientError

boto_session = Session()
region = boto_session.region_name
print(f"Using AWS region: {region}")

# Initialize the Bedrock AgentCore and SSM client
client = boto3.client('bedrock-agentcore', region_name=region)
ssm_client = boto3.client("ssm", region_name=region)


agent_arn_response = ssm_client.get_parameter(
        Name="/mcp_server/runtime_iam/agent_arn"
)

runtime_arn = agent_arn_response["Parameter"]["Value"]

print(f"Retrieved Agent ARN: {runtime_arn}")

if not runtime_arn:
        print("❌ Error: AGENT_ARN not found")
        sys.exit(1)
        
def call_mcp(method, params=None):
    """
    Call an MCP method on the agent runtime.
    
    Args:
        method: The MCP method to call (e.g., 'tools/list', 'tools/call')
        params: Optional parameters for the method
    
    Returns:
        The result from the MCP response
    """
    if params is None:
        params = {}

    payload = json.dumps({
        "jsonrpc": "2.0",
        "id": 1,
        "method": method,
        "params": params
    }).encode()

    try:
        response = client.invoke_agent_runtime(
            agentRuntimeArn=runtime_arn,
            payload=payload,
            qualifier='DEFAULT',
            contentType='application/json',
            accept='application/json, text/event-stream'
        )

        raw = response['response'].read().decode()
        json_data = json.loads(raw[raw.find('{'):])
        return json_data['result']

    except ClientError as e:
        print(f"\n{'=' * 60}")
        print("Error Response:")
        print(json.dumps(e.response, indent=2, default=str))
        print(f"{'=' * 60}\n")
        raise


def main():

    try:
        # List available tools
        print("📋 Available MCP Tools:")
        print("=" * 50)
        
        tools_result = call_mcp("tools/list")
        tools = tools_result['tools']
        
        for tool in tools:
            params = list(tool.get('inputSchema', {}).get('properties', {}).keys())
            print(f"🔧 {tool['name']}")
            print(f"   Description: {tool['description']}")
            print(f"   Parameters: {params}")
            print()
        
        print(f"✅ Successfully connected to MCP server!")
        print(f"Found {len(tools)} tools available.")

    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        import traceback

        print("\n🔍 Full error traceback:")
        traceback.print_exc()
        sys.exit(1)

if __name__ == "__main__":
    main()


### Testing Your Deployed MCP Server

Let's test our deployed MCP server using the remote client:

In [ ]:
print("Testing deployed MCP server...")
print("=" * 50)
!python mcp_client_remote_boto3.py

### Invoke MCP Tools - Boto3

Let's now use the boto3 SDK to create a client that not only lists tools but also invokes them:

In [ ]:
%%writefile invoke_mcp_tools_boto3.py
import sys
import boto3
import json
import logging
from boto3.session import Session
from botocore.exceptions import ClientError

boto_session = Session()
region = boto_session.region_name
client = boto3.client('bedrock-agentcore', region_name=region)

ssm_client = boto3.client("ssm", region_name=region)
agent_arn_response = ssm_client.get_parameter(Name="/mcp_server/runtime_iam/agent_arn")
runtime_arn = agent_arn_response["Parameter"]["Value"]

def call_mcp(method, params=None):
    if params is None:
        params = {}
    payload = json.dumps({
        "jsonrpc": "2.0",
        "id": 1,
        "method": method,
        "params": params
    }).encode()
    try:
        response = client.invoke_agent_runtime(
            agentRuntimeArn=runtime_arn,
            payload=payload,
            qualifier='DEFAULT',
            contentType='application/json',
            accept='application/json, text/event-stream'
        )
        raw = response['response'].read().decode()
        json_data = json.loads(raw[raw.find('{'):])
        # Return the full response, not just 'result' key
        return json_data
    except ClientError as e:
        print(f"❌ Error: {e}")
        raise

def main():

    print(f"Using AWS region: {region}")
    print(f"Retrieved Agent ARN: {runtime_arn}")


    print("\n🔄 Listing available tools...")
    try: 
        tools_response = call_mcp("tools/list")
        # Extract tools from the response
        tools_result = tools_response.get('result', {})

        print("\n📋 Available MCP Tools:")
        print("=" * 50)
        for tool in tools_result.get('tools', []):
            print(f"🔧 {tool['name']}: {tool['description']}")

        print("\n🧪 Testing MCP Tools:")
        print("=" * 50)

        # Test 1: Get server info
        print("\nTest 1: Getting Shotgrid server information...")
        try:
            response = call_mcp("tools/call", {
                "name": "get_server_info",
                "arguments": {}
            })
            print("✓ Server info retrieved successfully")
            
            # Extract data from result.content[0].text
            if 'result' in response and 'content' in response['result']:
                content = response['result']['content']
                if content and len(content) > 0:
                    text_content = content[0].get('text', '')
                    if text_content:
                        data = json.loads(text_content)
                        version = data.get('version', 'N/A')
                        # Format version as string if it's a list
                        if isinstance(version, list):
                            version = '.'.join(map(str, version))
                        url = data.get('analytics_site_name', 'N/A')
                        print(f"  - Server version: {version}")
                        print(f"  - Server URL: {url}")
        except Exception as e:
            print(f"✗ Error: {str(e)}")

        print()

        # Test 2: Find projects
        print("Test 2: Finding Shotgrid projects...")
        try:
            response = call_mcp("tools/call", {
                "name": "find_projects",
                "arguments": {"limit": 5}
            })
            print("✓ Projects retrieved successfully")
            
            # Extract projects from result.content
            if 'result' in response and 'content' in response['result']:
                content = response['result']['content']
                projects = []
                for item in content:
                    if 'text' in item:
                        try:
                            data = json.loads(item['text'])
                            if isinstance(data, dict):
                                projects.append(data)
                            elif isinstance(data, list):
                                projects.extend(data)
                        except json.JSONDecodeError:
                            pass
                
                if projects:
                    print(f"  - Found {len(projects)} project(s)")
                    for proj in projects[:5]:
                        name = proj.get('name', 'Unnamed')
                        proj_id = proj.get('id', 'N/A')
                        status = proj.get('sg_status', 'N/A')
                        print(f"    • {name} (ID: {proj_id}, Status: {status})")
        except Exception as e:
            print(f"✗ Error: {str(e)}")

        print()

        # Test 3: Find users
        print("Test 3: Finding Shotgrid users...")
        try:
            response = call_mcp("tools/call", {
                "name": "find_users",
                "arguments": {"limit": 5}
            })
            print("✓ Users retrieved successfully")
            
            # Extract users from result.content
            if 'result' in response and 'content' in response['result']:
                content = response['result']['content']
                users = []
                for item in content:
                    if 'text' in item:
                        try:
                            data = json.loads(item['text'])
                            if isinstance(data, dict):
                                users.append(data)
                            elif isinstance(data, list):
                                users.extend(data)
                        except json.JSONDecodeError:
                            pass
                
                if users:
                    print(f"  - Found {len(users)} user(s)")
                    for user in users[:5]:
                        name = user.get('name', 'Unnamed')
                        login = user.get('login', 'N/A')
                        status = user.get('sg_status_list', 'N/A')
                        print(f"    • {name} (Login: {login}, Status: {status})")
        except Exception as e:
            print(f"✗ Error: {str(e)}")

        print()


        print("\n" + "="*60)
        print("REMOTE TOOL INVOCATION TEST COMPLETE")
        print("="*60 + "\n")

    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        import traceback

        print("\n🔍 Full error traceback:")
        traceback.print_exc()
        sys.exit(1)

if __name__ == "__main__":
    main()


### Test Tool Invocation

Let's test our MCP tools by invoking our newly created client:

In [ ]:
print("Testing MCP tool invocation...")
print("=" * 50)
!python invoke_mcp_tools_boto3.py

## Agentic Scenario with Strands Agents SDK

So far we have tested the deployed MCP server by manually listing and invoking tools. In a real-world scenario, you would want an AI agent to autonomously decide which tools to call based on natural language instructions.

In this section we use the [Strands Agents SDK](https://strandsagents.com/) to create a fully agentic experience. The agent connects to the deployed Shotgrid MCP server on AgentCore Runtime via Streamable HTTP with SigV4 authentication, discovers the available tools, and uses Claude on Amazon Bedrock to reason about which tools to invoke.

This pattern follows the approach described in the [Strands Agents MCP integration workshop](https://catalog.workshops.aws/strands-agents/en-US/01-fundamentals/14-integration-mcp-tools).

### Architecture

```
User Prompt
    │
    ▼
┌──────────────────┐     Streamable HTTP + SigV4     ┌──────────────────────────┐
│  Strands Agent   │ ──────────────────────────────► │  Shotgrid MCP Server     │
│  (Claude on      │ ◄────────────────────────────── │  (AgentCore Runtime)     │
│   Bedrock)       │        Tool Results              │                          │
└──────────────────┘                                  └──────────────────────────┘
```

In [ ]:
# Install Strands Agents SDK
!pip install strands-agents strands-agents-tools --quiet

In [ ]:
%%writefile strands_shotgrid_agent.py
"""
Strands Agent with Shotgrid MCP Server on AgentCore Runtime.

This script creates a Strands agent that connects to the deployed Shotgrid MCP server
using Streamable HTTP transport with AWS SigV4 authentication, then uses Claude on
Amazon Bedrock to autonomously reason about and invoke Shotgrid tools.
"""

import boto3
from boto3.session import Session
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent
from strands.models.bedrock import BedrockModel
from strands.tools.mcp.mcp_client import MCPClient
from streamable_http_sigv4 import streamablehttp_client_with_sigv4


def create_sigv4_transport(mcp_url: str, region: str):
    """Create a Streamable HTTP transport with SigV4 auth for AgentCore Runtime."""
    session = boto3.Session()
    credentials = session.get_credentials()
    return streamablehttp_client_with_sigv4(
        url=mcp_url,
        credentials=credentials,
        service="bedrock-agentcore",
        region=region,
    )


def main():
    # --- Setup ---
    boto_session = Session()
    region = boto_session.region_name
    ssm_client = boto3.client("ssm", region_name=region)

    # Retrieve the agent ARN from Parameter Store (stored during deployment)
    agent_arn = ssm_client.get_parameter(Name="/mcp_server/runtime_iam/agent_arn")["Parameter"]["Value"]
    encoded_arn = agent_arn.replace(":", "%3A").replace("/", "%2F")
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"

    print(f"Region: {region}")
    print(f"Agent ARN: {agent_arn}")
    print(f"MCP URL: {mcp_url}\n")

    # --- Configure the Bedrock model ---
    model = BedrockModel(
        model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
        region_name=region,
    )

    # --- Connect to the MCP server via SigV4-authenticated Streamable HTTP ---
    mcp_client = MCPClient(lambda: create_sigv4_transport(mcp_url, region))

    with mcp_client:
        # Discover tools exposed by the Shotgrid MCP server
        tools = mcp_client.list_tools_sync()
        print(f"Discovered {len(tools)} tools from Shotgrid MCP server:\n")
        for tool in tools:
            desc = tool.tool_spec.get("description", "")[:80]
            print(f"  - {tool.tool_name}: {desc}...")
        print()

        # --- Create the Strands Agent ---
        system_prompt = (
            "You are a production management assistant for Autodesk Shotgrid. "
            "You help users query and manage their VFX/animation production data "
            "including projects, shots, assets, tasks, versions, and users. "
            "Use the available Shotgrid tools to answer questions. "
            "Always be concise and format results clearly."
        )

        agent = Agent(
            model=model,
            system_prompt=system_prompt,
            tools=tools,
        )

        # --- Run sample queries ---
        print("=" * 60)
        print("AGENTIC SHOTGRID INTERACTION")
        print("=" * 60)

        queries = [
            "What projects are available in Shotgrid? List the first 5.",
            "Can you get the Shotgrid server info and tell me what version it is running?",
        ]

        for query in queries:
            print(f"\nUser: {query}")
            print("-" * 40)
            response = agent(query)
            print(f"\n{'=' * 60}\n")


if __name__ == "__main__":
    main()


### Run the Strands Agent

Execute the agent script. The agent will connect to the deployed Shotgrid MCP server, discover the available tools, and use Claude to autonomously decide which tools to call based on the natural language queries:

In [ ]:
print("Running Strands Agent with Shotgrid MCP tools...")
print("=" * 50)
!python strands_shotgrid_agent.py

### Interactive Agent Session (Optional)

You can also run the agent interactively in the notebook. This lets you ask free-form questions and watch the agent reason about which Shotgrid tools to invoke:

In [ ]:
import boto3
from boto3.session import Session
from strands import Agent
from strands.models.bedrock import BedrockModel
from strands.tools.mcp.mcp_client import MCPClient
from streamable_http_sigv4 import streamablehttp_client_with_sigv4

boto_session = Session()
region = boto_session.region_name
ssm_client = boto3.client("ssm", region_name=region)

agent_arn = ssm_client.get_parameter(Name="/mcp_server/runtime_iam/agent_arn")["Parameter"]["Value"]
encoded_arn = agent_arn.replace(":", "%3A").replace("/", "%2F")
mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"

model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    region_name=region,
)

mcp_client = MCPClient(
    lambda: streamablehttp_client_with_sigv4(
        url=mcp_url,
        credentials=boto3.Session().get_credentials(),
        service="bedrock-agentcore",
        region=region,
    )
)

with mcp_client:
    tools = mcp_client.list_tools_sync()
    agent = Agent(
        model=model,
        system_prompt="You are a Shotgrid production management assistant. Use the available tools to help users.",
        tools=tools,
    )

    # Try your own query here:
    response = agent("List all active users in Shotgrid")

## Next Steps

Now that you have successfully deployed an MCP server to AgentCore Runtime, you can:

1. **Add More Tools**: Extend your MCP server with additional tools
2. **Custom Authentication**: Implement AWS IAM inbound authentication
3. **Integration**: Integrate with other AgentCore services

## Cleanup (Optional)

If you want to clean up the resources created during this tutorial, run the following cells:

In [ ]:
# try:
#     ssm_client.delete_parameter(Name='/mcp_server/runtime_iam/agent_arn')
#     print("✓ Parameter Store parameter deleted")
# except ssm_client.exceptions.ParameterNotFound:
#     print("ℹ️  Parameter Store parameter not found")

In [ ]:
# destroy_bedrock_agentcore(
#     config_path=Path(".bedrock_agentcore.yaml"),
#     agent_name=tool_name,
#     delete_ecr_repo=True
# )

# 🎉 Congratulations!

You have successfully:

✅ **Created an MCP server** with custom tools  
✅ **Tested locally** using MCP client  
✅ **Set up authentication** with Amazon Cognito  
✅ **Deployed to AWS** using AgentCore Runtime  
✅ **Invoked remotely** with proper authentication  
✅ **Learned MCP concepts** and best practices  

Your MCP server is now running on Amazon Bedrock AgentCore Runtime and ready for production use!

## Summary

In this tutorial, you learned how to:
- Build MCP servers using FastMCP
- Configure stateless HTTP transport for AgentCore compatibility
- Set up AWS IAM inbound authentication
- Deploy and manage MCP servers on AWS
- Test both locally and remotely
- Use MCP clients for tool invocation

The deployed MCP server can now be integrated into larger AI applications and workflows!